In [1]:
from box import ConfigBox



a = {'key1': 'value1', 'key2': 'value2'}
b= ConfigBox(a)

b.key1

'value1'

In [2]:
from ensure import ensure_annotations

@ensure_annotations
def test(a: int, b: int) -> int:
    return a + b

test(1, 2)

3

In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    
    

In [4]:
import os, sys
os.chdir("../")

In [5]:
%pwd


'a:\\Future\\MLOPS\\Projects\\Data science pro'

In [6]:
from src.DataSciencePro.constants import *
from src.DataSciencePro.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        param_file_path: Path = PARAM_FILE_PATH,
        scheme_file_path: Path = SCHEME_FILE_PATH,
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(param_file_path)
        self.schema = read_yaml(scheme_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir,
        )

        return data_ingestion_config

In [7]:
import os
import urllib.request as request
from src.DataSciencePro import logger
import zipfile


In [13]:
# Component Data ingestion

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, header= request.urlretrieve(
                url= self.config.source_URL,
                filename= self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with following info: \n{header}")
        else:
            logger.info(f"file already exists")
            
    def extract_zip_file(self):
        unzip_path= self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path) 
            

In [14]:
try:
    config= ConfigurationManager()
    data_ingestion_config= config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config= data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e
    

[2026-05-04 19:27:42,336 - DatascienceLogger - INFO - yaml file: config\config.yaml is loaded sucessfully.]
[2026-05-04 19:27:42,339 - DatascienceLogger - INFO - yaml file: params.yaml is loaded sucessfully.]
[2026-05-04 19:27:42,341 - DatascienceLogger - INFO - yaml file: schema.yaml is loaded sucessfully.]
Created directory at: artifacts
Created directory at: artifacts/data_ingestion
[2026-05-04 19:27:42,344 - DatascienceLogger - INFO - file already exists]
